# IndoBERT Fine-Tuning

Melakukan Fine-Tuning model `indobenchmark/indobert-base-p1` menggunakan Huggingface Transformers.

*Pastikan notebook ini dijalankan dengan GPU.*

In [1]:
!pip install -q transformers datasets torch scikit-learn

In [2]:
import datasets  # Import datasets first to avoid Windows DLL conflict with pyarrow
import pandas as pd
import numpy as np
import joblib
import torch
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix, precision_recall_fscore_support
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# Styling global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Load Data

In [3]:
train_df = pd.read_csv('../data/train.csv').fillna('')
val_df = pd.read_csv('../data/val.csv').fillna('')
test_df = pd.read_csv('../data/test.csv').fillna('')

X_train, y_train = train_df['text'].tolist(), train_df['label'].tolist()
X_val, y_val = val_df['text'].tolist(), val_df['label'].tolist()
X_test, y_test = test_df['text'].tolist(), test_df['label'].tolist()

le = joblib.load('../models/label_encoder.pkl')
num_classes = len(le.classes_)

## 2. Load Model & Tokenizer

In [4]:
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)

[transformers] You passed `num_labels=11` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 3. Buat Custom Dataset PyTorch

In [5]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(X_train, y_train, tokenizer)
val_dataset = NewsDataset(X_val, y_val, tokenizer)
test_dataset = NewsDataset(X_test, y_test, tokenizer)

## 4. Evaluasi Metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 5. Training

In [7]:
os.environ["TENSORBOARD_LOGGING_DIR"] = "../logs"
training_args = TrainingArguments(
    output_dir="../models/indobert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,                # Diubah dari 2e-5 menjadi 1e-5
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,                # Ditingkatkan ke 5 epoch (karena ada early stopping)
    weight_decay=0.05,                 # Ditingkatkan dari 0.01 ke 0.05
    load_best_model_at_end=True,
    metric_for_best_model="loss",      # Memantau val_loss untuk mencegah overfitting
    greater_is_better=False,
    logging_strategy="epoch",          # Log setiap epoch untuk history plot
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Menghentikan jika val_loss tidak membaik selama 2 epoch
)

print("Mulai Training IndoBERT...")
trainer.train()

Mulai Training IndoBERT...


c:\Users\Monk\miniconda3\envs\myenvironment\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 6. Visualisasi — Training History IndoBERT

In [ ]:
def plot_bert_training_history(trainer):
    """Plot training & eval loss/accuracy dari log Trainer HuggingFace."""
    logs = trainer.state.log_history

    # Pisahkan log training dan evaluasi
    train_logs = [l for l in logs if 'loss' in l and 'eval_loss' not in l]
    eval_logs  = [l for l in logs if 'eval_loss' in l]

    if not eval_logs:
        print("Tidak ada log evaluasi ditemukan. Pastikan training sudah selesai.")
        return

    train_epochs = [l.get('epoch', i+1) for i, l in enumerate(train_logs)]
    train_loss   = [l['loss'] for l in train_logs]

    eval_epochs  = [l['epoch'] for l in eval_logs]
    eval_loss    = [l['eval_loss'] for l in eval_logs]
    eval_acc     = [l.get('eval_accuracy', None) for l in eval_logs]
    eval_f1      = [l.get('eval_f1', None)       for l in eval_logs]

    has_acc = any(v is not None for v in eval_acc)
    n_plots = 3 if has_acc else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4.5))
    if n_plots == 1:
        axes = [axes]

    # --- Loss ---
    axes[0].plot(train_epochs, train_loss, 'o-', color='#1f77b4', label='Train Loss', linewidth=2)
    axes[0].plot(eval_epochs,  eval_loss,  's--', color='#ff7f0e', label='Val Loss',   linewidth=2)
    axes[0].set_title('IndoBERT — Training vs Validation Loss', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=9)
    axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    # Tandai best eval loss
    best_idx = int(np.argmin(eval_loss))
    axes[0].axvline(eval_epochs[best_idx], color='green', linestyle=':', linewidth=1.5,
                    label=f'Best Epoch {eval_epochs[best_idx]:.0f}')
    axes[0].legend(fontsize=9)

    if has_acc:
        # --- Accuracy ---
        clean_acc = [v if v is not None else float('nan') for v in eval_acc]
        axes[1].plot(eval_epochs, clean_acc, 's-', color='#2ca02c', label='Val Accuracy', linewidth=2)
        axes[1].set_title('IndoBERT — Validation Accuracy', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_ylim(0, 1.05)
        axes[1].legend(fontsize=9)
        axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

        # --- F1 ---
        clean_f1 = [v if v is not None else float('nan') for v in eval_f1]
        axes[2].plot(eval_epochs, clean_f1, 'd-', color='#d62728', label='Val F1 (weighted)', linewidth=2)
        axes[2].set_title('IndoBERT — Validation F1-Score', fontsize=11, fontweight='bold')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('F1-Score')
        axes[2].set_ylim(0, 1.05)
        axes[2].legend(fontsize=9)
        axes[2].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.suptitle('IndoBERT — Training History', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    os.makedirs('../outputs/figures', exist_ok=True)
    plt.savefig('../outputs/figures/indobert_training_history.png', bbox_inches='tight')
    plt.show()
    print('✓ Gambar disimpan: ../outputs/figures/indobert_training_history.png')

plot_bert_training_history(trainer)

## 7. Evaluasi ke Test Set

In [ ]:
print("Evaluasi pada Test Set:")
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

os.makedirs('../outputs', exist_ok=True)
indobert_results = pd.DataFrame([{'Model': 'IndoBERT', 'Accuracy': acc, 'F1-Score': f1}])
indobert_results.to_csv('../outputs/indobert_results.csv', index=False)

trainer.save_model("../models/indobert_final")

## 8. Visualisasi — Confusion Matrix IndoBERT

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, target_names):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f'Confusion Matrix — {model_name}', fontsize=14, fontweight='bold', y=1.01)

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=target_names, yticklabels=target_names,
        linewidths=0.5, linecolor='grey',
        ax=axes[0], cbar_kws={'shrink': 0.8}
    )
    axes[0].set_title('Jumlah Prediksi (Count)', fontsize=11)
    axes[0].set_xlabel('Prediksi')
    axes[0].set_ylabel('Aktual')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
        xticklabels=target_names, yticklabels=target_names,
        linewidths=0.5, linecolor='grey', vmin=0, vmax=1,
        ax=axes[1], cbar_kws={'shrink': 0.8}
    )
    axes[1].set_title('Proporsi per Kelas Aktual (Normalized)', fontsize=11)
    axes[1].set_xlabel('Prediksi')
    axes[1].set_ylabel('Aktual')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    os.makedirs('../outputs/figures', exist_ok=True)
    fname = f'../outputs/figures/cm_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f'  ✓ Gambar disimpan: {fname}\n')

plot_confusion_matrix(y_test, y_pred, 'IndoBERT', le.classes_)

## 9. Visualisasi — Metrik per Kelas IndoBERT

In [ ]:
prec, rec, f1_per, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=np.arange(len(le.classes_)), zero_division=0
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(le.classes_))
width = 0.25
color = '#9467bd'

bars_p = ax.bar(x - width, prec,   width, label='Precision', color=color, alpha=0.85)
bars_r = ax.bar(x,         rec,    width, label='Recall',    color=color, alpha=0.55)
bars_f = ax.bar(x + width, f1_per, width, label='F1-Score',  color=color, alpha=0.30)

for bar, val in zip(bars_f, f1_per):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('IndoBERT — Precision / Recall / F1 per Kelas', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.axhline(0.9, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='Threshold 0.90')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/indobert_per_class_metrics.png', bbox_inches='tight')
plt.show()
print('✓ Gambar disimpan: ../outputs/figures/indobert_per_class_metrics.png')

## 10. Visualisasi — Ringkasan Metrik Keseluruhan

In [ ]:
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted', zero_division=0
)
summary = {
    'Metric': ['Accuracy', 'Precision\n(weighted)', 'Recall\n(weighted)', 'F1-Score\n(weighted)'],
    'Score':  [acc, prec_w, rec_w, f1_w]
}
summary_df = pd.DataFrame(summary)

fig, ax = plt.subplots(figsize=(7, 4))
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax.bar(summary_df['Metric'], summary_df['Score'], color=palette, alpha=0.85, width=0.5)

for bar, val in zip(bars, summary_df['Score']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('IndoBERT — Ringkasan Metrik pada Test Set', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.axhline(0.9, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)

plt.tight_layout()
plt.savefig('../outputs/figures/indobert_summary_metrics.png', bbox_inches='tight')
plt.show()
print('✓ Gambar disimpan: ../outputs/figures/indobert_summary_metrics.png')
print(f"\nAccuracy : {acc:.4f}")
print(f"F1-Score : {f1_w:.4f}")